# Scheduling & Parameter Constraints Example

This notebook demonstrates additional features of the optimization framework including:
1. Learning rate scheduling
2. HSV channel callback scheduling
3. Dynamic augmentation callbacks


In [ ]:
import sys
import os
if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from torchvision.transforms.v2 import (
    GaussianBlur,
    Identity,
    RandomChoice,
    RandomHorizontalFlip,
    RandomPerspective,
    RandomResizedCrop,
)

from examples.example_scenes import BlenderManScene, CandleScene, CarScene, CarStudioScene, DinoScene, EinarScene, EinarSmallDomeScene, FlowerPotScene, HouseScene, RedCarScene, SciFiRobotScene, SpringPortraitScene, SpringPortraitSmallDomeScene, SpringScene
from losses.clip import CLIPDirectionalCosineSimilarity
from utils.color.linear_to_srgb_converters import LinearRec709ToAgXBase
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.model.model_utils import create_clip_model_and_tokenizer
from utils.optimize import optimize_with_criterion
from utils.parameter_strategies import HSVParameterStrategy


In [ ]:
# Select the scene to optimize (uncomment the desired scene)
scene = SciFiRobotScene(device=device)
# scene = SpringScene(device=device)
# scene = CarScene(device=device)
# scene = BlenderManScene(device=device)
# scene = RedCarScene(device=device)
# scene = CandleScene(device=device)
# scene = HouseScene(device=device)
# scene = DinoScene(device=device)
# scene = FlowerPotScene(device=device)
# scene = CarStudioScene(configuration='dome_lights', device=device)
# scene = EinarScene(device=device)
# scene = EinarSmallDomeScene(device=device)
# scene = SpringPortraitScene(device=device)
# scene = SpringPortraitSmallDomeScene(device=device)


In [ ]:
# Hyperparameters
lr = 0.04
n_iter = 250
global_seed = 2

A callback like this one can be passed to the `optimize_with_criterion` function to create a learning rate scheduler that wraps the chosen optimizer.

In [ ]:
# Learning Rate Scheduler Creator Callback
def learning_rate_scheduler_creator(optimizer: torch.optim.Optimizer) -> torch.optim.lr_scheduler.LRScheduler:
    n_warmup_steps = 50
    warmup = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: min((step + 1) / n_warmup_steps, 1.0)
    )
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_iter - n_warmup_steps, eta_min=1e-6
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[n_warmup_steps]
    )


Similarly, based on the epoch, an augmentation callback like this one can be passed to `optimize_with_criterion` to dynamically change the augmentation parameters during optimization.

In [ ]:
# Dynamic Augmentation Callback
def augmentation_callback(epoch: int):
    p_identity = max(0.0, 1.0 - (epoch / (n_iter * 0.75)))
    size = model.visual.preprocess_cfg["size"] or (224, 224) # type: ignore
    return RandomChoice([
        Identity(),
        RandomResizedCrop(size=size, scale=(0.4, 1.0), antialias=True), # type: ignore
        RandomPerspective(distortion_scale=0.2, p=1.0),
        RandomHorizontalFlip(p=1.0),
        GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.0)),
    ], p=[p_identity, (1.0 - p_identity) / 4, (1.0 - p_identity) / 4, (1.0 - p_identity) / 4, (1.0 - p_identity) / 4])


This is an exapmle of a callback that can be used to adjust which of the HSV channels of the RGB multipliers are free for optimization.

In [ ]:
# HSV Channel Scheduling Callback
def hsv_callback(epoch: int) -> str:
    if epoch < n_iter * 0.25:
        return "v"  # Only optimize brightness (Value) for first quarter
    elif epoch < n_iter * 0.5:
        return "sv"  # Saturation + Value
    else:
        return "hsv"  # All channels legal

In [ ]:
# Configure model
clip_model_name = "ViT-B-16-SigLIP-512"
clip_pretrained = "webli"

model, tokenizer, preprocess_eval = create_clip_model_and_tokenizer(
    clip_model_name,
    device=device,
    pretrained=clip_pretrained,
)


# Loss Criterion Setup
initial_text = "flat, unappealing lighting"
target_text = "golden hour sunset, warm glow"

color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
criterion = CLIPDirectionalCosineSimilarity(
    initial_text,
    target_text,
    scene.get_combined_image(color_space_converter).permute(2, 1, 0),
    model,
    tokenizer,
    device=device,
    preprocess=preprocess_eval,
)

title_prefix = "Scheduled HSV Optimization"

optimize_with_criterion(
    scene,
    lr,
    n_iter,
    criterion,
    starting_multiplier_std=(0.1, 0.1, 0.1),
    output_subdirectory_name="scheduling_example",
    n_results=4,
    learning_rate_scheduler_creator_callback=learning_rate_scheduler_creator,
    augmentation_callback=augmentation_callback,
    parameters_stored_as_hsv=True,
    hsv_callback=hsv_callback,
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix=title_prefix,
    device=device,
    save_every=50,
    model_name=clip_model_name,
    pretrained_source=clip_pretrained,
    seed=global_seed,
    show_images_after_augmentation=False,
)
